# Simstrat outputs analysis v2

This notebook is the proposed replacement for `Simstrat_outputs_analysis.ipynb`. It makes model loading explicit and uses the strict Kalden reader for parsed output times, input validation, depth normalization, altitude conversion, and pandas-3-safe resampling.


In [ ]:
from pathlib import Path

import pandas as pd

from kalden.core.simstrat import SimstratConfig, sum_complete_flows


## Configuration

Use one model dictionary as the source of truth. Removing or commenting a model never requires a later unconditional `pop`.


In [ ]:
MODEL_SETUPS = {
    "Etat actuel": Path(r"P:\Lausanne\CHA10063\03 Projet\03 Etudes de projet\02 Calculs\02_Simstrat\03_model_setups\Morat\v_20260608\base\Murten_Simstrat_config.par"),
    # Add comparison scenarios here.
}

LOAD_MODELS = False
OUTPUT_SEPARATOR = ","
OUTPUT_RESAMPLE = ""  # Keep exact timestamps by default; e.g. "1h" if needed.


## Strict model loading

The reader parses numeric or file-based output times. Missing inputs and malformed files raise by default rather than falling back to raw text. Input validation does not assume that an `Inflow` key exists.


In [ ]:
def load_model_set(
    setups: dict[str, Path],
    *,
    output_resample: str = OUTPUT_RESAMPLE,
) -> dict[str, SimstratConfig]:
    models = {}
    for name, setup_path in setups.items():
        model = SimstratConfig(setup_path)
        model.load_inputs(errors="raise", validate=True)
        model.load_outputs(
            sep=OUTPUT_SEPARATOR,
            resample=output_resample,
            errors="raise",
        )
        models[name] = model
    return models

def assert_time_integrity(model: SimstratConfig) -> None:
    for variable, frame in model.outputs.items():
        if not frame.index.is_monotonic_increasing:
            raise ValueError(f"{model.name}/{variable}: timestamps are not sorted.")
        if frame.index.has_duplicates:
            raise ValueError(f"{model.name}/{variable}: duplicate timestamps.")


## Depth and altitude access

Use `depths_to_altitudes` or `depth_to_altitude_table`. The compatibility property `depth_to_altitude` is also available, but new notebook code should use the canonical API.


In [ ]:
def output_coordinates(
    model: SimstratConfig,
    variable: str,
    *,
    use_altitudes: bool,
) -> pd.DataFrame:
    frame = model.outputs[variable]
    return model.depths_to_altitudes(frame) if use_altitudes else frame.copy()

def available_coordinates(
    model: SimstratConfig,
    *,
    use_altitudes: bool,
) -> list[float]:
    values = model.altitudes_output if use_altitudes else model.depths_output
    if values is None:
        coordinate = "altitudes" if use_altitudes else "depths"
        raise ValueError(f"No output {coordinate} are available for {model.name}.")
    return sorted((float(value) for value in values), reverse=True)


## Reusable one-dimensional comparison


In [ ]:
def plot_at_coordinate(
    models: dict[str, SimstratConfig],
    variable: str,
    coordinate: float,
    *,
    use_altitudes: bool = True,
):
    import plotly.graph_objects as go

    fig = go.Figure()
    for model_name, model in models.items():
        frame = output_coordinates(
            model,
            variable,
            use_altitudes=use_altitudes,
        )
        if coordinate not in frame.columns:
            raise KeyError(
                f"{model_name}/{variable}: coordinate {coordinate} is unavailable."
            )
        fig.add_scatter(
            x=frame.index,
            y=frame[coordinate],
            mode="lines",
            name=model_name,
        )
    axis_name = "altitude" if use_altitudes else "depth"
    fig.update_layout(title=f"{variable} at {axis_name} {coordinate:g}")
    return fig


## Scenario difference

Alignment is explicit. A difference is computed only on timestamps and coordinates shared by both scenarios.


In [ ]:
def scenario_difference(
    first: SimstratConfig,
    second: SimstratConfig,
    variable: str,
    *,
    use_altitudes: bool = True,
) -> pd.DataFrame:
    first_frame = output_coordinates(
        first, variable, use_altitudes=use_altitudes
    )
    second_frame = output_coordinates(
        second, variable, use_altitudes=use_altitudes
    )
    first_aligned, second_aligned = first_frame.align(
        second_frame,
        join="inner",
        axis=None,
    )
    if first_aligned.empty:
        raise ValueError("The selected scenarios have no common output data.")
    return first_aligned - second_aligned


## Water-balance totals

Use strict totals here as well; a missing inflow/outflow component is not a zero discharge.


In [ ]:
def total_model_flow(
    model: SimstratConfig,
    input_name: str,
    *,
    sign: float = 1.0,
) -> pd.Series:
    frame = model.inputs[input_name]
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{input_name!r} is not a loaded dataframe.")
    total = sum_complete_flows(frame, name=f"{input_name}_total")
    return total * sign

def daily_water_balance(model: SimstratConfig) -> pd.DataFrame:
    inflow = total_model_flow(model, "Inflow")
    outflow = total_model_flow(model, "Outflow", sign=-1.0)
    balance = pd.concat({"inflow": inflow, "outflow": outflow}, axis=1)
    return balance.resample("d").mean()


## Load and inspect


In [ ]:
models: dict[str, SimstratConfig] = {}
if LOAD_MODELS:
    models = load_model_set(MODEL_SETUPS)
    for model_name, model in models.items():
        assert_time_integrity(model)
        print(model_name)
        print(model.describe(print_output=False))
        print(f"Loaded outputs: {len(model.outputs)}")


## Migration map

- Move each stable analysis into a named function rather than relying on widget-created globals.
- Use `output_coordinates` for every depth/altitude conversion.
- Use `scenario_difference` before plotting scenario deltas.
- Use `sum_complete_flows` for hydrology and load calculations.
- Add derived lake science to a separate `kalden.analysis.simstrat` module with focused tests before porting the corresponding legacy cells.
